# Table S4. APC with the sugar-feed schedule of PPO or of the BO-tuned controller

APC (PAA set-point 1,200 mg/L from 10 h) was run on the 200 evaluation batches with the recipe F_s replaced by the
F_s schedule PPO chose for the same batch (ten seeds) or by the sugar-feed multiplier of the BO-tuned controller
- (ten runs). data/w78_apc_sugar.npz holds the yields (kg) and sugar fed (L of feed) of every run, and the yields of
APC, PPO and the BO-tuned controller on the same batches. Runs that failed to integrate are NaN.
Paired differences: mean over seeds or runs for each batch, then batch bootstrap (10,000 resamples).

In [1]:
import sys; sys.path.insert(0, '..')          # repository root, for paths.py
from paths import DATA, OUTD                   # data/ and output/

In [2]:
import numpy as np

z = np.load(DATA + 'w78_apc_sugar.npz')
Y = {'apc': z['y_apc'], 'apc_ppofs': z['y_apc_ppofs'], 'apc_bofs': z['y_apc_bofs']}
ref_apc, ppo, bo = z['y_apc_ref'], z['y_ppo'], z['y_bo']
rng = np.random.default_rng(0)


def ci(a, b):
    dd = np.nanmean(np.atleast_2d(a), 0) - np.nanmean(np.atleast_2d(b), 0); dd = dd[np.isfinite(dd)]
    bs = [rng.choice(dd, len(dd)).mean() for _ in range(10000)]
    return dd.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)


def sugar_kg(k):                       # glucose fed: L of sugar feed x 0.6 kg/L, mean over seeds per batch, then batches
    return np.nanmean(np.nanmean(np.atleast_2d(z[f'sugar_{k}']), 0)) * 0.6


L = ['Table S4. Penicillin yield and sugar fed on the 200 evaluation batches', '',
     f'{"Controller":44s}{"Penicillin (kg)":>16s}{"Sugar (kg)":>12s}',
     f'{"APC":44s}{np.nanmean(Y["apc"]):16,.0f}{sugar_kg("apc"):12,.0f}',
     f'{"APC with the PPO schedule of F_s":44s}{np.nanmean(Y["apc_ppofs"]):16,.0f}{sugar_kg("apc_ppofs"):12,.0f}',
     f'{"APC with the BO schedule of F_s":44s}{np.nanmean(Y["apc_bofs"]):16,.0f}{sugar_kg("apc_bofs"):12,.0f}',
     f'{"PPO":44s}{np.nanmean(ppo):16,.0f}{"(Table 2)":>12s}',
     f'{"BO-tuned controller":44s}{np.nanmean(bo):16,.0f}{"(Table 2)":>12s}',
     '', 'Paired difference in penicillin, mean [95 % bootstrap CI] (kg)']
for name, a, b in (('APC with the PPO schedule - APC', Y['apc_ppofs'], ref_apc),
                   ('APC with the BO schedule - APC', Y['apc_bofs'], ref_apc),
                   ('PPO - APC with the PPO schedule', ppo, Y['apc_ppofs']),
                   ('PPO - APC with the BO schedule', ppo, Y['apc_bofs']),
                   ('BO-tuned controller - APC with the BO schedule', bo, Y['apc_bofs'])):
    m, lo, hi = ci(a, b); L.append(f'  {name:48s}{m:+6.0f} [{lo:+.0f}, {hi:+.0f}]')
L.append(f'\nRuns that failed to integrate: ' + ', '.join(f'{k} {int(np.isnan(v).sum())}' for k, v in Y.items()))
txt = '\n'.join(L); print(txt); open(OUTD + 'tableS4.txt', 'w').write(txt + '\n')

Table S4. Penicillin yield and sugar fed on the 200 evaluation batches

Controller                                   Penicillin (kg)  Sugar (kg)
APC                                                    3,467      11,034
APC with the PPO schedule of F_s                       3,756      13,837
APC with the BO schedule of F_s                        3,685      13,152
PPO                                                    3,708   (Table 2)
BO-tuned controller                                    3,650   (Table 2)

Paired difference in penicillin, mean [95 % bootstrap CI] (kg)
  APC with the PPO schedule - APC                   +289 [+250, +325]
  APC with the BO schedule - APC                    +218 [+189, +244]
  PPO - APC with the PPO schedule                    -48 [-73, -29]
  PPO - APC with the BO schedule                     +23 [-10, +50]
  BO-tuned controller - APC with the BO schedule     -35 [-51, -21]

Runs that failed to integrate: apc 0, apc_ppofs 2, apc_bofs 4


981